<h1 style=\"text-align: center; font-size: 50px;\">  Text Generation with Neural Networks and Torch MLflow Integration</h1>

# Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2025-08-20 18:08:21 - INFO - Notebook execution started.


## User Constants

In [3]:
INITIAL_WORD = 'Love '
SIZE = 100

## Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

# Standard Library Imports
import warnings
from pathlib import Path

# Third-Party Libraries
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np

# MLflow for Experiment Tracking and Model Management
import mlflow
from mlflow.types.schema import Schema, ColSpec
from mlflow.models import ModelSignature

torch.manual_seed(0)

Note: you may need to restart the kernel to use updated packages.
CPU times: user 5.84 s, sys: 4.82 s, total: 10.7 s
Wall time: 5.9 s


## Configure Settings

In [5]:
warnings.filterwarnings("ignore")

In [6]:
# ------------------------ Define global experiment and run names to be used throughout the notebook ------------------------
EXPERIMENT_SET = "RNN text generation"
RUN_NAME = "RNN Text Generation"
MODEL_NAME = "dict_torch_rnn_model"
TORCH_MODEL = "dict_torch_rnn_model.pt"
REGISTER_NAME = "Shakespeare_Model"
EXPERIMENT_NAME = "Shakespeare Text Generation"

# ------------------------ Paths ------------------------
DATA_PATH = "../data/shakespeare.txt"
MODEL_DECODER_PATH = "models/decoder.pt"
MODEL_ENCODER_PATH = "models/encoder.pt"
MODEL_PATH = 'models/dict_torch_rnn_model.pt'
DEMO_FOLDER = "../demo"
CONFIG_PATH = "../configs/config.yaml"

## Verify Assets

In [7]:
def log_asset_status(asset_path: str, asset_name: str, success_message: str, failure_message: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
        success_message (str): Message to log if asset exists.
        failure_message (str): Message to log if asset does not exist.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured. {success_message}")
    else:
        logger.info(f"{asset_name} is not properly configured. {failure_message}")
        
log_asset_status(
    asset_path=DATA_PATH,
    asset_name="Shakespeare text",
    success_message="",
    failure_message="Please create and download the required assets in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_DECODER_PATH ,
    asset_name="Decoder model",
    success_message="",
    failure_message="Please check if model folder was properly downloaded in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_ENCODER_PATH,
    asset_name="Encoder model",
    success_message="",
    failure_message="Please check if model folder was properly downloaded in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_PATH,
    asset_name="Rnn model",
    success_message="",
    failure_message="Please check if model folder was properly downloaded in your project on AI Studio."
)
log_asset_status(
    asset_path=DEMO_FOLDER,
    asset_name="demo",
    success_message="",
    failure_message="Please check if demo folder was properly downloaded in your project on AI Studio."
)
log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="config",
    success_message="",
    failure_message="Please check if config file was properly downloaded in your project on AI Studio."
)

2025-08-20 18:08:27 - INFO - Shakespeare text is properly configured. 
2025-08-20 18:08:27 - INFO - Decoder model is properly configured. 
2025-08-20 18:08:27 - INFO - Encoder model is properly configured. 
2025-08-20 18:08:27 - INFO - Rnn model is properly configured. 
2025-08-20 18:08:27 - INFO - demo is properly configured. 
2025-08-20 18:08:27 - INFO - config is properly configured. 


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Creating the LSTM Model

In [9]:
# Get Text Data
with open(DATA_PATH,'r',encoding='utf8') as f:
    text = f.read()

In [10]:
all_characters = set(text) # creates a set of unique characters found in the text

In [11]:
class CharModel(nn.Module):
    def __init__(self, decoder, encoder, all_chars, num_hidden=256, num_layers=4,drop_prob=0.5, use_gpu=False):
        """Initializes CharModel

        Args:
            decoder: Assigns a unique integer to each character in a dictionary format
            encoder : Reverses the decoder dictionary, providing a mapping from characters to their respective assigned integers.
            all_chars: Set of unique characters found in the text.
            num_hidden: Number of hidden layers. Defaults to 256.
            num_layers: Number of layers. Defaults to 4.
            drop_prob: Regularization technique to prevent overfitting. Defaults to 0.5.
            use_gpu: If the model uses GPU. Defaults to False.
        """
        try:
            super().__init__()
            self.drop_prob = drop_prob
            self.num_layers = num_layers
            self.num_hidden = num_hidden
            self.use_gpu = use_gpu
            
            self.all_chars = all_chars
            self.decoder = torch.load(decoder)
            self.encoder = torch.load(encoder)
            
            self.lstm = nn.LSTM(len(self.all_chars), num_hidden, num_layers, dropout=drop_prob, batch_first=True)
            self.dropout = nn.Dropout(drop_prob)
            self.fc_linear = nn.Linear(num_hidden, len(self.all_chars))
            logger.info("CharModel initialized successfully")
    
        except Exception as e:
            logger.error(f"Error initializing CharModel: {str(e)}")
      
    
    def forward(self, x, hidden):
        """Implementation of the CharModel logic, in which, the input passes through every step of the arquiteture

        Args:
            x: Input tensor with shape (batch size and senquency length) containing character indices.
            hidden: Tuple containing the inicial hidden states of the CharModel each with shape (batch size and senquency length).

        Returns:
            final_out: Output tensor representing the predicted logits for each character in the sequence.
            hidden: Tuple containing the final hidden states of the CharModel.
        """
        try:
            lstm_output, hidden = self.lstm(x, hidden)       
            drop_output = self.dropout(lstm_output)
            drop_output = drop_output.contiguous().view(-1, self.num_hidden)
            final_out = self.fc_linear(drop_output)
            
            return final_out, hidden
        
        except Exception as e:
            logger.error(f"Error implementing CharModel logic: {str(e)}")
    
    
    def hidden_state(self, batch_size):
        """
        Initializes and returns the initial hidden state for a recurrent neural network (e.g., LSTM).

        This method creates zero-filled tensors for the hidden state (h_0) and cell state (c_0), 
        supporting GPU execution if `self.use_gpu` is set to True.

        Args:
            batch_size: The number of sequences in the input batch, used to determine the tensor dimensions.

        Returns:
            Tuple: A tuple containing the hidden state and cell state tensors 
            with shape (num_layers, batch_size, num_hidden). Returns None if an exception occurs, and logs the error.
        """
        try:
            if self.use_gpu:
                hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden).to(device),
                        torch.zeros(self.num_layers,batch_size,self.num_hidden).to(device))
            else:
                hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden),
                        torch.zeros(self.num_layers,batch_size,self.num_hidden))
            
            return hidden
        except Exception as e:
            logger.error(f"Error Initializing and returning the initial hidden state: {str(e)}")

## Logging Model to MLflow

In [ ]:
# Import the new Logger for models-from-code approach
from src.mlflow import Logger
from mlflow.types.schema import Schema, ColSpec
from mlflow.models import ModelSignature

# Define input/output schema for the RNN text generation model
input_schema = Schema([
    ColSpec("string", "initial_word"),
    ColSpec("long", "size")
])

output_schema = Schema([
    ColSpec("string", "generated_text")
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
logger.info("Model signature created successfully")

/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [13]:
mlflow.set_tracking_uri('/phoenix/mlflow')
mlflow.set_experiment(experiment_name= EXPERIMENT_NAME)

<Experiment: artifact_location='/phoenix/mlflow/144570119945864213', creation_time=1755712974630, experiment_id='144570119945864213', last_update_time=1755712974630, lifecycle_stage='active', name='Shakespeare Text Generation', tags={}>

In [14]:
model_state_dict = MODEL_PATH
register_name = REGISTER_NAME 

In [ ]:
with mlflow.start_run(run_name = RUN_NAME) as run:
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Use new Logger with models-from-code approach
    Logger.log_model(
        signature=signature,
        model_state_dict_path=MODEL_PATH,
        decoder_path=MODEL_DECODER_PATH,
        encoder_path=MODEL_ENCODER_PATH,
        artifact_path=model_state_dict,
        config_path=CONFIG_PATH,
        data_path=DATA_PATH,
        demo_folder=DEMO_FOLDER
    )
    
    mlflow.register_model(model_uri = f"runs:/{run.info.run_id}/{model_state_dict}", name=register_name)

2025-08-20 18:08:28 - INFO - Run's Artifact URI: /phoenix/mlflow/144570119945864213/7485baec2e2c4220b1ca5f2054ff5975/artifacts


2025-08-20 18:08:29 - INFO - Logging model to MLflow done successfully
Successfully registered model 'Shakespeare_Model'.
Created version '1' of model 'Shakespeare_Model'.


## Fetching the Latest Model Version from MLflow

In [16]:
client = mlflow.MlflowClient()
model_metadata = client.get_latest_versions(register_name, stages=["None"])
latest_model_version = model_metadata[0].version
latest_model_version

1

## Loading the Model and Running Inference

In [17]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{register_name}/{latest_model_version}")
print(model.predict({"initial_word": INITIAL_WORD, "size": SIZE}))

2025-08-20 18:08:29 - INFO - CharModel initialized successfully
2025-08-20 18:08:29 - ERROR - Error loading context: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



Love ||jvjj|vj||jvj|vv|vvvvvv||v|vvv|jjj|j|jvj|||jjjjv|vjj||vj|vjjjvv|vv||vv|||v|j||vj|j|||vvjjvj||||vjjjv


In [18]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-08-20 18:08:30 - INFO - ⏱️ Total execution time: 0m 9.02s
2025-08-20 18:08:30 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using Z by HP AI Studio.